<a href="https://colab.research.google.com/github/LailaBulh/Ingenieria_de_Datos_Avanzada/blob/main/Transformaciones_PySpark_LB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Transformaciones en PySpark**



**Nombre:** Laila Montserrat Bulhosen Ramos **Matrícula:** 263166

**Docente:** Dr. Vicente García Jiménez

**Materia:** Ingeniería de Datos Avanzada

**Link Github:** [Transformaciones_PySpark_LB](https://github.com/LailaBulh/Ingenieria_de_Datos_Avanzada/blob/main/Transformaciones_PySpark_LB.ipynb)

**Fecha:** Mayo 2026

### **Carga de librerías**

In [25]:
### Librería estándar de Python para interactuar con el sistema operativo
import os

### Clase para crear sesión en Spark
from pyspark.sql import SparkSession

# Módulo de funciones SQL de PySpark
# Se importa como F para facilitar su uso
from pyspark.sql import functions as F

# Tipos de datos utilizados para definir esquemas
from pyspark.sql.types import (
    StructType,      # Permite definir la estructura completa del DataFrame
    StructField,     # Permite definir cada columna del esquema
    StringType,      # Tipo cadena de texto
    IntegerType,     # Tipo entero
    DoubleType,      # Tipo numérico decimal
    TimestampType    # Tipo fecha y hora
)

# Librerías para visualización en Google Colab/Jupyter
from IPython.display import display, HTML

import pandas as pd

### **Instalación de PySpark**



In [26]:
### Verificar si PySpark está instalado

try:
  import pyspark

  print('PySpark ya esta instalado')
  print(f'Version de PySpark instalada: {pyspark.__version__}')


except ModuleNotFoundError:
    print("PySpark no está instalado. Instalando...")
    !pip install pyspark -q

    import pyspark
    print("Instalación completada")
    print("Versión:", pyspark.__version__)

PySpark ya esta instalado
Version de PySpark instalada: 4.0.2


In [27]:
### MOSTRAR VERSIÓN DESDE TERMINAL

print("\nInformación desde terminal:")
!pyspark --version


Información desde terminal:
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 4.0.2
      /_/
                        
Using Scala version 2.13.16, OpenJDK 64-Bit Server VM, 17.0.18
Branch HEAD
Compiled by user runner on 2026-02-02T08:08:13Z
Revision 7cc3b9bcdaab8c923f23cdbc9ce922530e1becf1
Url https://github.com/apache/spark
Type --help for more information.


### **Creación de SparkSession**

In [28]:
spark = (
    SparkSession.builder

    ### Nombre visible en la Spark UI y en los logs del clúster
    .appName('Transformaciones_PySpark')

    ### Número de particiones en operaciones de shuffle (default: 200)
    ### 200 particiones para datasets pequeños es excesivo y lento
    ### Regla general: 2-4 particiones por core en el clúster
    .config('spark.sql.shuffle.partitions', '8')

    ### Si ya existe una sesión activa, la reutiliza (no crea una nueva)
    .getOrCreate()
)

print(f'   SparkSession lista')
print(f'   Nombre app       : {spark.sparkContext.appName}')
print(f'   Master           : {spark.sparkContext.master}')
print(f'   Cores disponibles: {spark.sparkContext.defaultParallelism}')

   SparkSession lista
   Nombre app       : Transformaciones_PySpark
   Master           : local[*]
   Cores disponibles: 2


### **1. Carga de archivo CSV**

In [29]:
### Drive connection

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
### Dirección donde se ubica el archivo en drive
ruta_csv = '/content/drive/MyDrive/NSL_KDD.csv'

### Cargar el archivo CSV en un DataFrame de PySpark
df_kdd = (
    spark.read

    ### Indica que el archivo contiene encabezados
    .option('header', 'true')

    ### Aplicar esquema automáticamente
    .option('inferSchema', 'true')

    ### Cargar archivo CSV
    .csv(ruta_csv)
)

In [31]:
### Mostrar esquema del dataframe

df_kdd.printSchema()

root
 |-- duration: string (nullable = true)
 |-- protocol_type: string (nullable = true)
 |-- service: string (nullable = true)
 |-- flag: integer (nullable = true)
 |-- src_bytes: integer (nullable = true)
 |-- dst_bytes: integer (nullable = true)
 |-- land: integer (nullable = true)
 |-- wrong_fragment: integer (nullable = true)
 |-- urgent: integer (nullable = true)
 |-- hot: integer (nullable = true)
 |-- num_failed_logins: integer (nullable = true)
 |-- logged_in: integer (nullable = true)
 |-- num_compromised: integer (nullable = true)
 |-- root_shell: integer (nullable = true)
 |-- su_attempted: integer (nullable = true)
 |-- num_root: integer (nullable = true)
 |-- num_file_creations: integer (nullable = true)
 |-- num_shells: integer (nullable = true)
 |-- num_access_files: integer (nullable = true)
 |-- num_outbound_cmds: integer (nullable = true)
 |-- is_host_login: integer (nullable = true)
 |-- is_guest_login: integer (nullable = true)
 |-- count: integer (nullable = true

Al ver los resultados del Schema podemos ver que la columna '*duration'* es tipo string y no númerico.

En el paso 4 de esta actividad se pide filtrar por *protocol_type = 'tcp'*, sabiendo esto e investigando sobre el dataset, se puede comprobar que los valores en la columna de '*duration*' en realidad corresponden a la columna '*protocol_type*'.

Esto se puede ver también en la columna de '*service*' donde los datos de esta columna se encuentran en la columna '*ptrotocol_type*'.

Para corregir esto es necesario renombrar las columnas de manera correcta y actualizar el dataframe con el que se trabajará.

También se pide trabajar con la columna '*label*' por lo que es necesario renombrar la columna que aparece originalmente como '*class*'.

In [32]:
### Nuevos nombres de columnas

new_col_names = [ 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
                  'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins',
                  'logged_in', 'num_compromised', 'root_shell', 'su_attempted',
                  'num_root', 'num_file_creations', 'num_shells', 'num_access_files',
                  'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count',
                  'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate',
                  'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
                  'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
                  'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
                  'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
                  'dst_host_serror_rate', 'dst_host_srv_serror_rate',
                  'dst_host_rerror_rate', 'dst_host_srv_rerror_rate',
                  'label', 'difficulty'
                  ]


* Se crea *df_updated* para guardar el dataset con las columnas renombradas

* Con *.toDF()* se utilizan los mismos datos pero con nombres de columnas diferentes

* *new_col_names* es la lista que contiene los nuevos nombres de columnas y con * se expande y pasa cada nombre como argumento separado.

In [33]:
### Creación de dataframe con los nombres de columna actualizadas

df_updated = df_kdd.toDF(*new_col_names)

### Se verifica el esquema nuevamente
df_updated.printSchema()

root
 |-- protocol_type: string (nullable = true)
 |-- service: string (nullable = true)
 |-- flag: string (nullable = true)
 |-- src_bytes: integer (nullable = true)
 |-- dst_bytes: integer (nullable = true)
 |-- land: integer (nullable = true)
 |-- wrong_fragment: integer (nullable = true)
 |-- urgent: integer (nullable = true)
 |-- hot: integer (nullable = true)
 |-- num_failed_logins: integer (nullable = true)
 |-- logged_in: integer (nullable = true)
 |-- num_compromised: integer (nullable = true)
 |-- root_shell: integer (nullable = true)
 |-- su_attempted: integer (nullable = true)
 |-- num_root: integer (nullable = true)
 |-- num_file_creations: integer (nullable = true)
 |-- num_shells: integer (nullable = true)
 |-- num_access_files: integer (nullable = true)
 |-- num_outbound_cmds: integer (nullable = true)
 |-- is_host_login: integer (nullable = true)
 |-- is_guest_login: integer (nullable = true)
 |-- count: integer (nullable = true)
 |-- srv_count: integer (nullable = tru

In [34]:
### Verificación de cambios

df_updated.show(5)

+-------------+-------+----+---------+---------+----+--------------+------+---+-----------------+---------+---------------+----------+------------+--------+------------------+----------+----------------+-----------------+-------------+--------------+-----+---------+-----------+---------------+-----------+---------------+-------------+-------------+------------------+--------------+------------------+----------------------+----------------------+---------------------------+---------------------------+--------------------+------------------------+--------------------+------------------------+------------+----------+
|protocol_type|service|flag|src_bytes|dst_bytes|land|wrong_fragment|urgent|hot|num_failed_logins|logged_in|num_compromised|root_shell|su_attempted|num_root|num_file_creations|num_shells|num_access_files|num_outbound_cmds|is_host_login|is_guest_login|count|srv_count|serror_rate|srv_serror_rate|rerror_rate|srv_rerror_rate|same_srv_rate|diff_srv_rate|srv_diff_host_rate|dst_host_

### **2. Información general del dataframe**

In [35]:
### Número total de filas

print(f'Número total de filas en dataset KDD: {df_updated.count():,} registros')

Número total de filas en dataset KDD: 18,035 registros


In [36]:
### Número total de columnas

print(f'Número total de columnas en dataset KDD: {len(df_updated.columns)}')

Número total de columnas en dataset KDD: 42


In [37]:
### Primeras filas

df_updated.show(10)

+-------------+-------+----+---------+---------+----+--------------+------+---+-----------------+---------+---------------+----------+------------+--------+------------------+----------+----------------+-----------------+-------------+--------------+-----+---------+-----------+---------------+-----------+---------------+-------------+-------------+------------------+--------------+------------------+----------------------+----------------------+---------------------------+---------------------------+--------------------+------------------------+--------------------+------------------------+------------+----------+
|protocol_type|service|flag|src_bytes|dst_bytes|land|wrong_fragment|urgent|hot|num_failed_logins|logged_in|num_compromised|root_shell|su_attempted|num_root|num_file_creations|num_shells|num_access_files|num_outbound_cmds|is_host_login|is_guest_login|count|srv_count|serror_rate|srv_serror_rate|rerror_rate|srv_rerror_rate|same_srv_rate|diff_srv_rate|srv_diff_host_rate|dst_host_

### **3. Selección de columnas**


In [38]:
### Dataframe para guardar la selección de columnas del dataframe original
df_select = df_updated.select('protocol_type','service','src_bytes','dst_bytes','label')

df_select.show(10)

+-------------+-------+---------+---------+------------+
|protocol_type|service|src_bytes|dst_bytes|       label|
+-------------+-------+---------+---------+------------+
|          tcp|private|        0|        0|     neptune|
|          tcp|private|        0|        0|     neptune|
|          tcp| telnet|        0|       15|       mscan|
|          tcp|   http|      267|    14515|      normal|
|          tcp| telnet|      129|      174|guess_passwd|
|          tcp|   http|      327|      467|      normal|
|          tcp|    ftp|       26|      157|guess_passwd|
|          tcp| telnet|        0|        0|       mscan|
|          tcp|private|        0|        0|     neptune|
|          tcp| telnet|        0|        0|     neptune|
+-------------+-------+---------+---------+------------+
only showing top 10 rows


### **4. Filtro de registros**

Se pide filtrar por dos condiciones, la columna protocol_type' sea igual a 'tcp'  y que los valores de la columna 'src_bytes' sean mayores a 1000 por lo que se crea el siguiene dataframe.


In [41]:
df_filter = df_select.filter(
                            (F.col('protocol_type') == 'tcp')
                            & (F.col('src_bytes') > 1000)
                            )

df_filter.show()


+-------------+--------+---------+---------+-----------+
|protocol_type| service|src_bytes|dst_bytes|      label|
+-------------+--------+---------+---------+-----------+
|          tcp|    http|    76944|        1|    apache2|
|          tcp|ftp_data|   283618|        0|warezmaster|
|          tcp|    http|    72564|        0|    apache2|
|          tcp|    smtp|     2599|      293|   mailbomb|
|          tcp|    smtp|     4030|      332|     normal|
|          tcp|    http|    72564|        0|    apache2|
|          tcp|    http|    54540|     8314|       back|
|          tcp|    smtp|     1376|      343|     normal|
|          tcp|    smtp|     1500|      332|     normal|
|          tcp|    smtp|     1710|      366|     normal|
|          tcp|ftp_data|   283618|        0|warezmaster|
|          tcp|    http|    54540|     8315|       back|
|          tcp|ftp_data|     6932|        0|     normal|
|          tcp|    smtp|     1055|      364|     normal|
|          tcp|ftp_data|   2836

### **5. Creación nuevas columnas**

Se agrega una nueva columna 'total_bytes' con la sumatoria de las columnas 'src_bytes' y 'dst_bytes'.

In [42]:
df_newCol = df_filter.withColumn('total_bytes', F.col('src_bytes') + F.col('dst_bytes'))

df_newCol.show()

+-------------+--------+---------+---------+-----------+-----------+
|protocol_type| service|src_bytes|dst_bytes|      label|total_bytes|
+-------------+--------+---------+---------+-----------+-----------+
|          tcp|    http|    76944|        1|    apache2|      76945|
|          tcp|ftp_data|   283618|        0|warezmaster|     283618|
|          tcp|    http|    72564|        0|    apache2|      72564|
|          tcp|    smtp|     2599|      293|   mailbomb|       2892|
|          tcp|    smtp|     4030|      332|     normal|       4362|
|          tcp|    http|    72564|        0|    apache2|      72564|
|          tcp|    http|    54540|     8314|       back|      62854|
|          tcp|    smtp|     1376|      343|     normal|       1719|
|          tcp|    smtp|     1500|      332|     normal|       1832|
|          tcp|    smtp|     1710|      366|     normal|       2076|
|          tcp|ftp_data|   283618|        0|warezmaster|     283618|
|          tcp|    http|    54540|

### **6. Organizar datos de manera descendete por la columna '*total_bytes*'**

In [43]:
### Dataframe para almacenar el groupby()

df_ordered = df_newCol.orderBy(F.col('total_bytes').desc())

df_ordered.show(10, truncate = False)


### Se imprime el total de resutados obtenidos
total_row = df_newCol.count()
print('\nEl total de resultados encontrados es:', total_row)

+-------------+--------+---------+---------+------+-----------+
|protocol_type|service |src_bytes|dst_bytes|label |total_bytes|
+-------------+--------+---------+---------+------+-----------+
|tcp          |X11     |62825648 |90476    |xlock |62916124   |
|tcp          |X11     |31645608 |207796   |xlock |31853404   |
|tcp          |ftp_data|6291668  |0        |normal|6291668    |
|tcp          |ftp_data|3131464  |0        |normal|3131464    |
|tcp          |ftp_data|2194619  |0        |normal|2194619    |
|tcp          |X11     |13948    |1171108  |normal|1185056    |
|tcp          |X11     |286040   |383476   |normal|669516     |
|tcp          |X11     |39224    |511712   |named |550936     |
|tcp          |ftp_data|501760   |0        |normal|501760     |
|tcp          |ftp_data|501760   |0        |normal|501760     |
+-------------+--------+---------+---------+------+-----------+
only showing top 10 rows

El total de resultados encontrados es: 1552


### **7. Valores únicos de '*protocol_type*'**

In [44]:
df_unique = df_updated.select('protocol_type').distinct()

df_unique.show( truncate = False)

+-------------+
|protocol_type|
+-------------+
|udp          |
|tcp          |
|icmp         |
+-------------+



### **8. Agrupar por la columna '*protocol_type* ' agregando la función count()**

In [46]:
df_grouped_pt = df_updated.groupBy('protocol_type').count()

### Se muestran de manera descendente los resultados de la agrupación
df_grouped_pt.orderBy(F.col('count').desc()).show( truncate = False)

+-------------+-----+
|protocol_type|count|
+-------------+-----+
|tcp          |15136|
|udp          |2074 |
|icmp         |825  |
+-------------+-----+



### **9. Agrupar por columna '*label* ' y cálculo de promedio**

In [48]:
df_grouped_label = df_updated.groupBy('label'
                                      ).agg(F.avg('src_bytes'). alias('src_bytes_avg')
                                      )

### Verificación de total de resultados para imprimir los resultados
### completos en el siguiente paso

print(df_grouped_label.count())

38


In [49]:
### Se muestra el total de filas luego de agrupar

df_grouped_label.show(40, truncate = False)

+---------------+-------------------+
|label          |src_bytes_avg      |
+---------------+-------------------+
|buffer_overflow|1867.0             |
|multihop       |1767.8333333333333 |
|normal         |2501.925664398511  |
|warezmaster    |65186.258530183724 |
|portsweep      |0.0                |
|xsnoop         |775.75             |
|httptunnel     |506.83653846153845 |
|smurf          |909.5143403441682  |
|snmpguess      |45.456204379562045 |
|ipsweep        |14.666666666666666 |
|xterm          |3137.909090909091  |
|xlock          |1.350409942857143E7|
|ps             |124.58333333333333 |
|sqlattack      |398.0              |
|neptune        |0.0                |
|satan          |0.5208333333333334 |
|sendmail       |1698.7272727272727 |
|worm           |4209.0             |
|rootkit        |54713.2            |
|land           |0.0                |
|pod            |1341.2121212121212 |
|processtable   |0.03052064631956912|
|mailbomb       |2599.0             |
|named      

### **Cerrar sesión de Spark**

Como buena práctica de programación, se cierra la sesión de Spark evitando consumos innecesarios de memoria.

In [22]:
spark.stop()

print('✅ SparkSession finalizada correctamente.')

✅ SparkSession finalizada correctamente.


### **Referencia**

* Kaggle.com. Retrieved May 16, 2026, from https://www.kaggle.com/datasets/kiranmahesh/nslkdd